# 4.4 Een telling vóór en na een gebeurtenis (event)

[04.2](04.2-long-tail%20jeffrey.ipynb) fitte één verdelingsfamilie op één populatie; [04.3](04.3-penguins%20jeffrey.ipynb) liet zien wat een fit doet wanneer de populatie in feite uit meerdere groepen bestaat. Beide beschreven een statische situatie zoals die is. Dit notebook beantwoordt de vraag waar de les naartoe heeft gebouwd: **heeft een gebeurtenis (event) het onderliggende proces veranderd, en zo ja, wélke parameter is verschoven?** We bouwen voort op de fitter en haar twee winnaars uit 04.2, de les over samengevoegd versus gegroepeerd uit 04.3, en de release-dag indicator uit les 03.1.

Alles tot nu toe beschreef steeds één situatie. De methodologische stap die van een verdelingsfit hard bewijs maakt, is het fitten van dezelfde familie aan weerszijden van een vooraf bekend event, om vervolgens af te lezen wat er veranderd is. Het event hier is hetzelfde als in les 03.1: Ubuntu-releases, tien donderdagen tussen 2013 en 2017, in de twee kanalen van de showcase `ubuntu_irc_release_hourly` — `#ubuntu` (het grote hoofdkanaal) en `#ubuntu-it`. De variabele is de eenvoudigste discrete telling die er is: **berichten per donderdag**. De showcase arriveert als uurtotalen met de release-dagen reeds gemarkeerd, dus de telling is één `GroupAgg` per kanaal.


In [ ]:
import pandas as pd

from goad_toolkit.analytics import DistributionFitter, FitResult, NullDistribution, fit_table
from goad_toolkit.datatransforms import Filter, GroupAgg, Pipeline
from goad_toolkit.distributions import DistributionRegistry
from goad_toolkit.visualizer import DistPlot, ECDFPlot, HistogramPlot, NullPlot, PlotSettings

from wa_analyzer.data import load_own_chat, load_showcase

fitter = DistributionFitter(DistributionRegistry(), seed=42)

In [ ]:
hourly = load_showcase("ubuntu_irc_release_hourly")


def messages_per_day(channel: str) -> pd.DataFrame:
    """One row per Thursday: total messages that day, and whether it was a release day."""
    return (
        Pipeline()
        .add(Filter, expr=f"channel == '{channel}'")
        .add(GroupAgg, by=["date", "is_release"], column="messages", agg="sum")
        .apply(hourly)
    )


it_days = messages_per_day("#ubuntu-it")
it_ordinary = it_days.loc[~it_days.is_release, "messages"].to_numpy().astype(float)
it_release = it_days.loc[it_days.is_release, "messages"].to_numpy().astype(float)
print(f"#ubuntu-it: {len(it_ordinary)} ordinary Thursdays, {len(it_release)} release days")
print(f"ordinary: mean {it_ordinary.mean():.0f}, variance {it_ordinary.var():.0f}")

## 4.4.1 Welke familie past op een gewone donderdag?

Berichten per dag is een discrete telling, dus de klassieke leerboek-hypothese is **Poisson**: onafhankelijke gebeurtenissen die binnenkomen met een constant tempo (*steady rate*). Poisson doet een toetsbare belofte — haar variantie is exact gelijk aan haar gemiddelde. De bovenstaande printout breekt die belofte al met een factor van enkele honderden, dus de fit hoort te falen. *Hóe* de fit faalt is de kernles. Fit elke discrete familie en teken de twee belangrijkste over de data.


In [ ]:
it_fits = fitter.fit(it_ordinary, discrete=True)
fit_table(it_fits)[["distribution", "params", "log_likelihood", "ks_pvalue", "best_likelihood", "best_ks"]]

In [ ]:
by_name = {f.distribution: f for f in it_fits if isinstance(f, FitResult)}

steady = PlotSettings(
    figsize=(10, 4),
    title="#ubuntu-it, ordinary Thursdays: a steady rate is the wrong hypothesis",
    xlabel="messages per day",
    ylabel="density",
)
host = HistogramPlot(steady)
fig, ax = host.plot(data=it_ordinary, bins=30, color="lightgrey")
host.plot_on(DistPlot(steady), distribution=by_name["poisson"].frozen_dist,
             color="crimson", label="poisson: one steady rate")
_ = host.plot_on(DistPlot(steady), distribution=by_name["nbinom"].frozen_dist,
                 x_range=(0, it_ordinary.max()), color="steelblue", label="nbinom: a rate that wanders")

De Poisson-verdeling is een vlijmscherpe naald. Met een gemiddelde van ongeveer 560 staat het dagelijkse schommelingen toe van grofweg ±25 berichten; in werkelijkheid schommelt het kanaal tussen minder dan 100 en meer dan 1.500 berichten. De KS p-waarde in de tabel is nul tot op vijftig decimalen nauwkeurig. Die verwerping is geen mislukking van de fit — het is de ontdekking: **er is helemaal geen sprake van een constant tempo.** Wélke donderdag het is doet ertoe: het jaartal (het kanaal kromp over de tijd), wie er toevallig online was, wat er die week kapot ging.

De **negatief-binomiale verdeling** (*negative binomial* / `nbinom`) is exact de familie hiervoor: een Poisson-verdeling waarvan het tempo zelf elke dag willekeurig wordt getrokken. Haar twee parameters zijn een gemiddelde (*mean*) en een dispersie (*dispersion* — hoe ver het tempo uitzwermt). Het vormt de bovenste rij van de tabel, met een KS p-waarde die de verdeling niet verwerpt. De eerlijke samenvatting van een gewone donderdag is dus niet "ongeveer 560 berichten", maar "een tempo van ongeveer 560 dat routinematig halveert of verdrievoudigt". Dat is opnieuw reden 1: een beschrijving die daadwerkelijk past.

## 4.4.2 Dezelfde familie, op release-dagen

Tien release-dagen is een kleine steekproef, maar de vraag is ook specifiek: blijft de verdelingsfamilie overeind, en welke parameter is verschoven? Fit de verdeling en toon beide steekproeven op één as zonder verstorende bins.


In [ ]:
release_fits = fitter.fit(it_release, discrete=True)
release_best = next(f for f in release_fits if isinstance(f, FitResult) and f.best_likelihood)
ordinary_best = by_name["nbinom"]

print(f"ordinary Thursday: {ordinary_best.distribution}, mean {ordinary_best.frozen_dist.mean():.0f}, "
      f"sd {ordinary_best.frozen_dist.std():.0f}")
print(f"release day:       {release_best.distribution}, mean {release_best.frozen_dist.mean():.0f}, "
      f"sd {release_best.frozen_dist.std():.0f}")
print(f"rate on a release day: x{it_release.mean() / it_ordinary.mean():.2f}")

fig, ax = ECDFPlot(PlotSettings(figsize=(8, 4), title="#ubuntu-it: ordinary Thursdays vs release days",
                      xlabel="messages per day", ylabel="share of days at or below")).plot(
    data=it_ordinary, compare=it_release, label="ordinary Thursday", compare_label="release day",
)

Zelfde familie aan weerszijden, en wat verschoven is, is het **tempo** (*rate*): een release-dag draait op ruwweg 1,4× een gewone donderdag, met een vergelijkbare spreiding eromheen. De ECDF toont exact hetzelfde zonder een enkele bin-keuze: de rode curve ligt over vrijwel de gehele lengte rechts van de blauwe curve — release-dagen zijn verschoven (*shifted*), niet van fundamentele vorm veranderd (*not reshaped*). Die zin, *het tempo verschoof en de vorm bleef gelijk*, is wat fitten aan weerszijden oplevert ten opzichte van het simpelweg vergelijken van twee gemiddelden.

## 4.4.3 Hoe zeker mogen we zijn?

Een toename van 1,4× op basis van tien dagen, getrokken uit een verdeling die "routinematig halveert of verdrievoudigt", zou puur toeval kunnen zijn. De controle is degene waar deze cursus steeds naar terugkeert: **hoe zou dit getal eruitzien als het label release/gewoon betekenisloos was?** `NullDistribution` schudt het `is_release`-label willekeurig over de donderdagen (*permutation test*), berekent telkens het tempobereik opnieuw, en herhaalt dat duizend keer. `NullPlot` tekent deze wolk van toevalsuitkomsten en markeert de werkelijke geobserveerde waarde daarin.


In [ ]:
def rate_difference(days: pd.DataFrame) -> float:
    """Mean messages on release days minus mean on ordinary Thursdays."""
    means = days.groupby("is_release").messages.mean()
    return means[True] - means[False]


null = NullDistribution(rate_difference, n_iter=2000, seed=4)
it_null = null.run(it_days, label="is_release")

fig, ax = NullPlot(PlotSettings(figsize=(8, 4), title="#ubuntu-it: is a +230 lift more than chance?",
                                xlabel="release − ordinary, messages per day")).plot(result=it_null)
ax.legend()
print(f"two-sided p: {it_null.p_value():.3f}   one-sided p (release days busier): {it_null.p_value('greater'):.3f}")

De waargenomen toename bevindt zich in de rechterbovenstaart van de wolk, maar valt er niet buiten — een tweezijdige p-waarde van ongeveer 0,07 (eenzijdig, "drukker", circa 0,04). Tien dagen bij een tempo dat zo sterk fluctueert vormt een smalle basis, en het getal laat dat eerlijk zien.

Nu dezelfde drie stappen op `#ubuntu`, het kanaal met tienmaal zoveel verkeer.


In [ ]:
ubuntu_days = messages_per_day("#ubuntu")
ubuntu_null = null.run(ubuntu_days, label="is_release")

for name, days, result in [("#ubuntu-it", it_days, it_null), ("#ubuntu", ubuntu_days, ubuntu_null)]:
    ordinary = days.loc[~days.is_release, "messages"]
    lift = days.loc[days.is_release, "messages"].mean() / ordinary.mean()
    print(f"{name:11s} lift x{lift:.2f}   ordinary-day spread sd/mean = {ordinary.std() / ordinary.mean():.2f}"
          f"   two-sided p = {result.p_value():.4f}")

compare = PlotSettings(
    figsize=(13, 4),
    title="The same lift, two verdicts",
    subplot_titles=["#ubuntu-it: rate x1.4, borderline", "#ubuntu: rate x1.5, unmistakable"],
    xlabel="release − ordinary, messages per day",
    ylabel="density",
    max_cols=2,
)
host = NullPlot(compare)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(NullPlot(compare), axes[0], result=it_null)
host.plot_on_axes(NullPlot(compare), axes[1], result=ubuntu_null)
for ax in axes:
    ax.legend()

Beide kanalen tonen een release-toename van ongeveer anderhalf. In `#ubuntu` is het effect onmiskenbaar: de werkelijke waarde staat ver buiten alles wat de geschudde labels produceren. In `#ubuntu-it` is het een grensgeval. **Het effect heeft dezelfde omvang; het statistische bewijs niet.** De printout verklaart waarom: een gewone donderdag in `#ubuntu` wijkt gemiddeld ongeveer een derde af van haar gemiddelde, in `#ubuntu-it` ongeveer twee derde. Wanneer vele onafhankelijke individuen uitmiddelen, worden dagen meer aan elkaar gelijk — 04.1 §4.1.2 opnieuw, sommen van vele bijdragen — waardoor dezelfde relatieve stijging in het grote kanaal helder boven de ruis uitsteekt, terwijl het in het kleine kanaal verdrinkt in de dagelijkse schommelingen.

Dat is de les over steekproefomvang in haar meest bruikbare vorm: wat je kunt detecteren is het effect *gedeeld door* de spreiding van de variabele die je meet, en die spreiding is een eigenschap van je data die je al kunt aflezen aan de gewone dagen vóórdat je überhaupt naar het event kijkt. Een student die eerst de gewone donderdagen fit, weet vooraf al of tien event-dagen toereikend zijn om de hypothese te beslechten.


## 4.4.4 Jouw beurt — vóór en na, op je eigen WhatsApp-chat

`load_own_chat()` geeft een foutmelding als er nog geen export is — een doe-het-zelf sectie zonder data valt niet te toetsen. Mocht de volgende cel vastlopen, draai dan eerst [01.3](../lesson1/01.3-your-own-chat%20jeffrey.ipynb) en verwijs in `config.toml` naar het geëxporteerde bestand.


In [ ]:
own = load_own_chat()
own["timestamp"] = pd.to_datetime(own["timestamp"])

### Denk eerst in verdelingsfamilies

Voordat je iets fit, beslis welk type variabele je onderzoekt en wat een verandering daarin concreet betekent — de familie vertelt je wélke parameter je moet aflezen:

| Variabele in je chat | Familie | Wat een verandering betekent |
|---|---|---|
| Berichten per dag | Poisson als het tempo constant is; **nbinom** wanneer het fluctueert (dat doet het altijd) | Het **tempo** (*rate*) is verschoven, of de **dispersie** (*dispersion*) — drukker, of grilliger |
| Seconden tussen berichten, binnen een burst | **exponential** | De **schaal** (*scale*, gemiddelde pauze) is verschoven — snellere of tragere interactie |
| Berichten per dag per persoon | Poisson / nbinom, één per auteur | *Wie* er veranderde, niet alleen hoeveel |

Een enkele fit over je hele chat is zelden interessant: "berichten per dag is overgedispergeerd" geldt voor elke chat (geen enkele vriendengroep houdt jarenlang een vast tempo aan), en "berichtlengte is lognormaal" geldt voor elke chat om de redenen uit [04.2](04.2-long-tail%20jeffrey.ipynb). De interessante variant is wat §4.4.1–§4.4.3 zojuist demonstreerde: **een bekend event splitst de data in tweeën, en de fit aan weerszijden vertelt exact wát er veranderde.** In les 03.3 heb je zo'n event benoemd; gebruik het hier.


In [ ]:
my_event = "2025-09-21"
my_event_end = "2025-10-06"
per_day = (
    own.set_index("timestamp").resample("D").size().rename("messages").reset_index()
)
# We vergelijken de 16 reisdagen (True) met alle gewone dagen thuis (False)
per_day["after"] = (
    (per_day["timestamp"] >= pd.Timestamp(my_event, tz=per_day["timestamp"].dt.tz)) & 
    (per_day["timestamp"] <= pd.Timestamp(my_event_end, tz=per_day["timestamp"].dt.tz))
)
print(per_day.groupby("after").messages.agg(["count", "mean", "var"]).round(1))

**Berichten per dag, aan weerszijden.** Fit de discrete families aan beide kanten van het event. Lees twee dingen af aan de parameters, niet één: verschoof het *gemiddelde*, en verschoof de *dispersie*? Een groep kan exact hetzelfde aantal berichten sturen, maar op een veel grilligere manier — dat is een wezenlijk ander analytisch inzicht.


In [ ]:
fitter = DistributionFitter(DistributionRegistry(), seed=42)
for label, side in [("before", per_day[~per_day.after]), ("after", per_day[per_day.after])]:
    fits = fitter.fit(side.messages.to_numpy().astype(float), discrete=True)
    best = next(f for f in fits if isinstance(f, FitResult) and f.best_likelihood)
    print(f"{label:7s} {best.distribution:8s} mean {best.frozen_dist.mean():6.1f}   "
          f"sd {best.frozen_dist.std():6.1f}   ({len(side)} days)")

In [ ]:
own_null = NullDistribution(
    lambda days: days.groupby("after").messages.mean().diff().iloc[-1], n_iter=2000, seed=4,
).run(per_day, label="after")

fig, ax = NullPlot(PlotSettings(figsize=(8, 4), title=f"Messages per day: after {my_event} minus before",
                                xlabel="after − before, messages per day")).plot(result=own_null)
_ = ax.legend()

**Pauzes tussen berichten, aan weerszijden.** De exponentiële verdeling is de familie voor wachttijden bij een constant tempo, en "binnen een actieve conversatie-uitbarsting (burst)" is de enige plek waar een chat zoiets als een stabiel tempo kent. De pauzes worden daarom afgeknipt op één uur, en de twee fits worden vergeleken op hun *schaalparameter* (de gemiddelde tussentijd).

⚠️ Dit vereist tijdstempels met een resolutie van seconden. IRC-logs registreren enkel `[HH:MM]`, waardoor meer dan de helft van opeenvolgende berichten in dezelfde minuut valt en de pauze per definitie nul is — een meetfout, geen modelleringsfout. WhatsApp-exports bevatten wél seconden; de onderstaande controle weigert te fitten als jouw data geen seconden bevat.


In [ ]:
ts = own["timestamp"].sort_values()
on_the_minute = ((ts.dt.second == 0) & (ts.dt.microsecond == 0)).mean()
if on_the_minute > 0.3:
    print(f"{on_the_minute:.0%} of timestamps land exactly on the minute: too coarse for a gap fit.")
else:
    gaps = pd.DataFrame({"gap": ts.diff().dt.total_seconds(), "timestamp": ts}).dropna()
    gaps = gaps[(gaps.gap > 0) & (gaps.gap < 3600)]  # within an hour: a burst, not overnight
    gaps["after"] = gaps.timestamp >= pd.Timestamp(my_event, tz=gaps.timestamp.dt.tz)
    for label, side in [("before", gaps[~gaps.after]), ("after", gaps[gaps.after])]:
        fit = fitter.fit_distribution("exponential", side.gap.to_numpy())
        print(f"{label:7s} mean gap {fit.frozen_dist.mean():6.0f} s   ({len(side):,} gaps)")  # ty: ignore[unresolved-attribute]

In [ ]:
# =============================================================================
# 1. DATA-FILTERING & DAGTOTALEN (BERICHTEN PER DAG: THUISBASIS VS. THAILAND)
# =============================================================================
# Doel van deze cel:
# Aggregeren van chatberichten naar dagtotalen over de volledige tijdsspanne.
# Cruciaal: We gebruiken .resample('D') zodat rustige dagen zonder berichten (0 berichten)
# expliciet in de dataset worden opgenomen, wat essentieel is voor een zuivere kansverdeling.

from pathlib import Path
import numpy as np
import pandas as pd
from wa_analyzer.data import ROOT, load_own_chat

# Laad de geëxporteerde en verrijkte WhatsApp-data
own = load_own_chat()
own["timestamp"] = pd.to_datetime(own["timestamp"])

# Dagelijkse berichtentelling inclusief nul-dagen
daily = (
    own.set_index("timestamp")
    .resample("D")
    .size()
    .rename("messages")
    .reset_index()
)

# Periode-afbakening van de reis naar Thailand
TH_START = "2025-09-21"
TH_END = "2025-10-06"

daily["is_thailand"] = (daily["timestamp"] >= TH_START) & (
    daily["timestamp"] <= TH_END
)

th_days = daily[daily.is_thailand].copy()
base_days = daily[~daily.is_thailand].copy()

n_days_th = len(th_days)
n_days_base = len(base_days)

# Gemiddelden en uitersten berekenen
mean_base = base_days["messages"].mean()
mean_th = th_days["messages"].mean()
diff_mean = mean_th - mean_base

# Percentage rustige dagen (<= 3 berichten) in de normale situatie
quiet_days_base = (base_days["messages"] <= 3).mean() * 100
peak_th = int(th_days["messages"].max())

print("Data-verwerking gereed:")
print(f"  Thuisbasis (Baseline) : {n_days_base:,} dagen, gem. {mean_base:.1f} ber/dag ({quiet_days_base:.0f}% ≤ 3 berichten)".replace(",", "."))
print(f"  Thailand-reis         : {n_days_th} dagen, gem. {mean_th:.1f} ber/dag (piekdag: {peak_th} berichten)")
print(f"  Verschil in tempo     : +{diff_mean:.1f} berichten per dag")


In [ ]:
# =============================================================================
# 2. STATISTISCHE BEWIJSLAST & FITTING (NEGATIEF-BINOMIAAL MODEL & PERMUTATIETOETS)
# =============================================================================
# Doel van deze cel:
# Wiskundige modellering en statistische toetsing van het verschil in berichtentempo:
# 1. Waarom Negatief-Binomiaal (nbinom)?
#    Chatberichten vertonen sterke clustering (overdispersie: variantie >> gemiddelde).
#    Een Poisson-verdeling eist var = mean en onderschat de zware staart catastrofaal.
# 2. Permutatietoets (N = 2.000 iteraties):
#    Toetsen of het waargenomen verschil (+22,2 ber/dag) door toeval kan ontstaan.

from goad_toolkit.analytics import DistributionFitter
from goad_toolkit.distributions import DistributionRegistry

# Fitten van de discrete nbinom-verdeling op beide steekproeven
fitter = DistributionFitter(DistributionRegistry(), seed=42)
fit_base = fitter.fit_distribution("nbinom", base_days.messages.to_numpy().astype(float))
fit_th = fitter.fit_distribution("nbinom", th_days.messages.to_numpy().astype(float))

# Kansdichtheidscurven (PMF) berekenen over het bereik van 0 t/m 105 berichten
x_support = np.arange(0, 106)
y_base = fit_base.frozen_dist.pmf(x_support)
y_th = fit_th.frozen_dist.pmf(x_support)

# Permutatietoets: hussel alle dagen willekeurig en meet hoe vaak een verschil >= waargenomen optreedt
rng = np.random.default_rng(42)
all_messages = daily["messages"].to_numpy()
observed_diff = diff_mean

perm_diffs = []
for _ in range(2000):
    perm = rng.permutation(all_messages)
    perm_th_mean = perm[:n_days_th].mean()
    perm_base_mean = perm[n_days_th:].mean()
    perm_diffs.append(perm_th_mean - perm_base_mean)

p_perm = (np.array(perm_diffs) >= observed_diff).mean()
p_str = "p < 0,001" if p_perm < 0.001 else f"p = {p_perm:.3f}".replace(".", ",")

# Dynamische variabelen voor de visualisatie in cel 3
mean_base_str = f"{mean_base:.1f}".replace(".", ",")
mean_th_str = f"{mean_th:.1f}".replace(".", ",")
diff_mean_str = f"{diff_mean:.1f}".replace(".", ",")

print("STATISTISCHE BEWIJSLAST (Modelfits & Permutatietoets):")
print(f"  Fit Thuisbasis : nbinom(n={fit_base.params.n:.2f}, p={fit_base.params.p:.3f})")
print(f"  Fit Thailand   : nbinom(n={fit_th.params.n:.2f}, p={fit_th.params.p:.3f})")
print(f"  Permutatietoets: Waargenomen tempoverschil = +{diff_mean_str} ber/dag, {p_str} (N = 2.000 iteraties)")
print("  Conclusie      : Het verschil is uiterst significant; toeval is statistisch uitgesloten.")


In [ ]:
# =============================================================================
# 3. VISUALISATIE (KANSDICHTHEID & MODELCURVEN CONFORM TUFTE & SCHWABISH)
# =============================================================================
# Doel van deze cel:
# Presenteren van de empirische data (staven) en wiskundige modellen (lijnen):
# - Grey-first: rustig leisteengrijs voor de thuisbasis, levendig terracotta voor Thailand.
# - Badges en annotaties direct op het canvas gevoed vanuit cel 1 en 2.
# - 2-laags header en methodologische voetnoot zonder hardcoded getallen.

import matplotlib.pyplot as plt
from scripts.plots import (
    JeffreyPlotSettings,
    annotate_badge,
    apply_dav_theme,
    format_footnote,
    format_header,
)

# 1. Huisstijl en canvas
settings = JeffreyPlotSettings(figsize=(11.8, 6.0))
fig, ax = plt.subplots(figsize=settings.figsize)

apply_dav_theme(ax)

# 2. Histogramstaven (werkelijke relatieve dichtheid per bakje van 3 berichten)
bins = np.arange(0, 108, 3)
ax.hist(
    base_days.messages,
    bins=bins,
    density=True,
    color="#CBD5E0",
    alpha=0.55,
    edgecolor="white",
    zorder=2,
    label="Thuisbasis (werkelijk)",
)
ax.hist(
    th_days.messages,
    bins=bins,
    density=True,
    color="#FDBA74",
    alpha=0.60,
    edgecolor="white",
    zorder=2,
    label="Thailand (werkelijk)",
)

# 3. Gefitte Negatief-Binomiale verdelingscurven
ax.plot(
    x_support,
    y_base,
    color=settings.base_color,
    linewidth=3.2,
    zorder=3,
    label="Fit: Thuisbasis",
)
ax.plot(
    x_support,
    y_th,
    color=settings.highlight_color,
    linewidth=3.2,
    zorder=3,
    label="Fit: Thailand",
)

# 4. Assen en raster
ax.set_xlim(-1, 106)
ax.set_ylim(0, 0.26)
ax.set_xlabel(
    "Aantal WhatsApp-berichten per dag",
    fontsize=12,
    fontweight="bold",
    color=settings.dark_text,
    labelpad=10,
)
ax.set_ylabel(
    "Dichtheid (Density)",
    fontsize=12,
    fontweight="bold",
    color=settings.dark_text,
    labelpad=10,
)
ax.grid(axis="y", color="#E2E8F0", linestyle="--", linewidth=1.0, alpha=0.7)

# 5. Drie prominente annotatie-badges (dynamisch gevoed met berekende variabelen)
annotate_badge(
    ax,
    text=f"Thuisbasis:\n{quiet_days_base:.0f}% van de dagen ≤ 3 berichten\n(hoge dichtheid bij nul, daalt steil)",
    xy=(1.5, 0.178),
    xytext=(20.0, 0.22),
    highlight=False,
    settings=settings,
)

annotate_badge(
    ax,
    text=f"Thailand-reis:\nDichtheid vlakt af en spreidt ver uit\n(gemiddelde verschuift van {mean_base_str} naar {mean_th_str})",
    xy=(26.0, 0.016),
    xytext=(40.0, 0.09),
    highlight=True,
    settings=settings,
)

annotate_badge(
    ax,
    text=f"Piekdag Thailand:\n{peak_th} berichten",
    xy=(float(peak_th), 0.005),
    xytext=(72.0, 0.05),
    highlight=True,
    settings=settings,
)

# 6. Header en dynamische methodologische voetnoot
n_base_fmt = f"{n_days_base:,}".replace(",", ".")

format_header(
    ax,
    title="Van radiostilte naar berichtenregen: Thailand blaast stille WhatsApp-chat op",
    subtitle=f"Kansdichtheid van dagelijkse berichten: Thuisbasis ({n_base_fmt} dagen, grijs) versus Thailand-reis ({n_days_th} dagen, oranje)",
)

footnote_text = (
    f"Werkelijke chatdagen (staven) gemodelleerd als dichtheid tegen de wiskundige nbinom-verdeling (lijnen): thuisbasis gem. {mean_base_str} ber/dag → Thailand gem. {mean_th_str} ber/dag.\n"
    f"Met {p_str} (Permutatietoets met 2.000 iteraties) is toeval uitgesloten: de reis naar Thailand veranderde ons appgedrag compleet."
)
format_footnote(ax, text=footnote_text, y_offset=-0.18)

# 7. Opslaan en tonen
output_img = ROOT / "img" / "figuur3_verdelingen_voor_na.png"
output_img.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output_img, dpi=200, bbox_inches="tight")
plt.show()


### Wat je moet noteren (voor de inlevering)

1. **Het event en wat je verwachtte dat zou veranderen** — tempo (*rate*), dispersie, of de schaal van pauzes — opgeschreven vóórdat je de cellen draaide. Citeer jezelf letterlijk.
2. **De twee fits**, als een volwaardige zin met de parameters erin: "vóór het event, nbinom met gemiddelde 31 en sd 19; na het event, gemiddelde 52 en sd 40". Een tabel is prima; een los screenshot van een histogram volstaat niet.
3. **De nulplot, en waar de werkelijke waarde zich bevindt.** Als de waarde midden in de toevalswolk valt, zeg dat dan eerlijk — een verandering die niet te onderscheiden is van toeval is een waardevol inzicht over je steekproefomvang, en §4.4.3 liet zien hoe je dat vooraf al kunt voorspellen uit de spreiding van de normale dagen.

Dit is exact waar de checklist van de `goad` MCP (`goad_analysis_checklist`) je op zal bevragen; een voor/na-onderzoeksvraag met een benoemd event is precies de vorm die het verwacht.

## Reflectievragen

1. Kies één familie uit [04.1](04.1-parametric-distributions%20jeffrey.ipynb) en noem een variabele in je eigen chat waarvan je verwacht dat die deze verdeling volgt. Wat moet er waar zijn over *hoe die variabele tot stand komt* om die verwachting waar te maken — en waardoor zou de fit kunnen falen zoals Poisson faalde in §4.4.1?
2. In [04.2](04.2-long-tail%20jeffrey.ipynb) veranderde de log-transformatie een "uitbijter" in een volkomen alledaagse waarneming. Waar in je eigen data zou het omgekeerde gebeuren — een logaritme die een normaal punt juist verdacht maakt?
3. §4.4.3 vond dezelfde relatieve toename met twee totaal verschillende oordelen. Voor je eigen event: hoeveel event-dagen had je, puur op basis van de spreiding van de gewone dagen, nodig gehad om zeker te kunnen zijn?
4. Jouw event splitst de chat in de *tijd*; [04.3](04.3-penguins%20jeffrey.ipynb) splitste op basis van *groepen*. Als de mensen in je chat tijdens het event veranderden van samenstelling (*wie er aanwezig was*), over welke van de twee splitsingen gaat de ontdekking dan in werkelijkheid?


---

**Waar dit nu naartoe gaat.** Elke fit in deze les tot nu toe was op ruwe data zoals die binnenkwam. Het meest waardevolle om een verdeling op te fitten is echter wat een *model* heeft achtergelaten: [04.5-fitting-the-residual](04.5-fitting-the-residual%20jeffrey.ipynb) fit een rechte lijn, trekt deze af, en vraagt aan het residu of het model klaar is of een wezenlijk mechanisme over het hoofd ziet.
